# From English to an Alpaca paper-trading strategy
### Week 4 · SeeBurtOh

For my Week 4 project, I chose Ed's idea of generating trading code against a simulated equities API.
I wanted to see whether a model could turn a plain-English rule into a small Python function that I
could test before connecting it to Alpaca.

I started with a 10/30-day moving-average rule and compared three models. **GPT-6 Luna is my default**:
it passed the same checks as Astra at a much lower cost in this experiment. Generation makes paid
OpenAI API calls. The notebook starts with `GENERATE_WITH_LLM = False` and uses the reference
implementation so the default run needs no API credentials. I enable generation explicitly.

**English rule → generated Python → behavior checks → backtest → paper-order preview**

The notebook is my experiment log. `strategy_lab.py` handles generation and backtesting,
`paper_broker.py` connects to Alpaca, and `gradio_app.py` provides an optional local interface.
I keep account access and order submission off until I explicitly enable them.

## 1. Environment and configuration

I use the course's Python environment as the notebook kernel. The course already includes OpenAI,
NumPy, pandas, matplotlib, python-dotenv, and Gradio; Alpaca's SDK is the extra dependency.
The commented install cell below is only needed if that SDK is missing.

The project has been checked with Python 3.12, `openai==2.7.1`, `alpaca-py==0.45.0`, and
`gradio==5.49.1`. My API credentials stay in the repository root `.env`.

In [ ]:
# Run once if Alpaca is not installed in this kernel:
# %pip install "alpaca-py==0.45.0"

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Code, display
from dotenv import load_dotenv

# Support launching Jupyter from either the repository root or this folder.
HERE = Path.cwd()
if not (HERE / "strategy_lab.py").exists():
    HERE = HERE / "week4/community-contributions/SeeBurtOh/alpaca_codegen"
if not (HERE / "strategy_lab.py").exists():
    raise RuntimeError(
        "Launch Jupyter from the repository root or this contribution folder."
    )
sys.path.insert(0, str(HERE))
REPO = next(p for p in HERE.parents if (p / "pyproject.toml").exists())
load_dotenv(REPO / ".env", override=False)

from strategy_lab import (  # noqa: E402
    API_CONTRACT,
    DEFAULT_SPEC,
    REFERENCE_SOURCE,
    Strategy,
    StrategyError,
    acceptance_cases,
    check_cases,
    generate_strategy,
    synthetic_bars,
    backtest,
)

GENERATE_WITH_LLM = False  # Set True to generate with GPT-6 Luna (paid API).
USE_ALPACA_DATA = False  # Explicit opt-in: reads Alpaca historical data.
CONNECT_PAPER = False  # Explicit opt-in: reads account and previews an order.
MODEL = "gpt-6-luna"  # Explicit model choice; independent of OPENAI_MODEL.
SYMBOL = "AAPL"  # Example software input, not a stock recommendation.
print(
    f"Generation: {MODEL} (paid API)"
    if GENERATE_WITH_LLM
    else "Generation: authored reference (offline)"
)

## 2. The strategy API

I give the model one function to implement: `signal(closes)`. The input is a list of daily closing
prices, oldest first. The result is **1 to hold a long position** or **0 to hold cash**. My host code
turns changes in that decision into buys or sells and handles position sizing.

This separation lets me use the same generated function in a historical test and a paper account.
The generated code never receives credentials or calls the broker directly.

`Strategy` reads a limited subset of Python's syntax tree: local assignments, arithmetic, comparisons,
conditionals, price slices, and a few builtins. It rejects imports, loops, attributes, and arbitrary
calls. It does not use `exec` or `eval`. This is a deliberately small interpreter, not a sandbox for
general Python programs.

In [ ]:
print(API_CONTRACT)
print("Strategy request:", DEFAULT_SPEC)
display(Code(REFERENCE_SOURCE, language="python"))
reference = Strategy(REFERENCE_SOURCE)
check_cases(reference, acceptance_cases())
print("Reference passed the warmup, tie, rising, falling, and reversal examples.")

## 3. Generation, validation, and repair

My default is `MODEL = "gpt-6-luna"` with low reasoning effort. `GENERATE_WITH_LLM = True` calls the
OpenAI API using `OPENAI_API_KEY` from `.env`. The reference path and local backtests make no model calls.
[GPT-6 Luna pricing](https://developers.openai.com/api/docs/models/gpt-6-luna)

The prompt contains the API contract and the English rule. I check the returned code against the
contract and expected examples, then allow one repair attempt. I record every attempt, including
failures, timing, and token usage. The reference implementation is separate from generated results.

If I change the rule, I also change `CASES`. The expected answers need to come from the specification;
a model should not decide whether its own answer is correct.

In [ ]:
SPECIFICATION = DEFAULT_SPEC
CASES = acceptance_cases()
experiment = pd.DataFrame()
if GENERATE_WITH_LLM:
    from openai import OpenAI

    client = OpenAI(timeout=60.0, max_retries=0)
    candidate, experiment = generate_strategy(client, MODEL, SPECIFICATION, CASES)
    display(experiment.drop(columns="source"))
    if candidate is None:
        raise RuntimeError(
            "Both attempts failed. Inspect experiment['source'] and revise the prompt."
        )
    provenance = f"LLM-generated using {MODEL}"
else:
    candidate = reference
    provenance = "Authored reference (offline demo; no model called)"
print(provenance)
display(Code(candidate.source, language="python"))
print("Source fingerprint:", candidate.digest)

## 4. Independent behavior checks

Parsing successfully does not tell me whether the model implemented the right rule. For this
moving-average example, I compare its answers with an independent calculation on 150 randomized
histories that were not sent to the model. A different rule would need a different oracle.

I also check that imports, attribute access, loops, and invalid signals are rejected. These tests
exercise the interface; they do not prove that every possible generated strategy is correct.

In [ ]:
rng = np.random.default_rng(2026)
if SPECIFICATION == DEFAULT_SPEC:
    for length in range(0, 150):
        history = np.exp(rng.normal(4.6, 0.1, length)).tolist()
        expected = (
            int(np.mean(history[-10:]) > np.mean(history[-30:])) if length >= 30 else 0
        )
        assert candidate(history) == expected, f"Mismatch at history length {length}"
    print("150 independent behavioral checks passed.")

rejected = [
    "import os\ndef signal(closes):\n    return 1",
    "def signal(closes):\n    return closes.__class__",
    "def signal(closes):\n    while 1:\n        pass",
    "def signal(closes):\n    return 2",
]
for source in rejected:
    try:
        Strategy(source)([100.0] * 40)
    except StrategyError:
        continue
    raise AssertionError("Unsupported or invalid behavior was accepted")
print("All four rejection checks passed.")

## 5. Synthetic data and Alpaca setup

I start with reproducible synthetic prices containing trends, noise, and overnight gaps. The dates
are weekdays rather than an exchange calendar. This fixture tests the implementation without an
account; its returns are not observations about actual stocks.

My setup for Alpaca is:
1. Sign in or create an account at [Alpaca](https://app.alpaca.markets/account/login).
2. Select **Paper Trading**, then generate a key and secret in the dashboard's API Keys panel.
   A paper-only account uses simulated funds; I do not need to fund a live account.
3. Add the paper credentials to the existing repository root `.env` using these exact variable names:
   ```dotenv
   ALPACA_PAPER_API_KEY=your_paper_key
   ALPACA_PAPER_SECRET_KEY=your_paper_secret
   ```
4. Restart the kernel after saving the file. The loader preserves existing environment variables,
   so a restart also avoids retaining old credentials after a key change.
5. Enable `USE_ALPACA_DATA` for the historical-data cell below, or choose **Alpaca IEX** in Gradio.

[Alpaca's setup guide](https://alpaca.markets/learn/start-paper-trading) explains the account and key steps.
My adapter requests split-adjusted daily IEX bars and excludes today's incomplete bar. IEX represents
one exchange rather than the whole market. The backtest does not model every corporate action.
[Market data reference](https://alpaca.markets/sdks/python/api_reference/data/stock/requests.html)

In [ ]:
if USE_ALPACA_DATA:
    from paper_broker import PaperBroker

    data_broker = PaperBroker()
    end = pd.Timestamp.now(tz="America/New_York").normalize()
    bars = data_broker.bars(SYMBOL, end - pd.Timedelta(days=730), end)
    data_label = f"{SYMBOL}: Alpaca IEX daily bars, split-adjusted"
else:
    bars = synthetic_bars(seed=42)
    data_label = "Synthetic data, seed=42 (not actual market observations)"
print(data_label)
display(bars.head())
print(f"{len(bars)} bars, {bars.index[0]} through {bars.index[-1]}")

## 6. Chronological backtesting

I calculate the signal after day **t** closes and execute at **t+1's open**, with assumed fees and
slippage. Using the same close for both the decision and the fill would give the strategy execution
that it could not necessarily achieve.

The engine buys whole shares using 20% of cash when flat, then holds until an exit signal. It does
not short, borrow, or force a final liquidation. Open holdings are valued at the last close. My
buy-and-hold comparison uses the same entry allocation and costs, rather than a fully invested account.

I reserve the last 35% of dates as a holdout. Each period starts with a fresh $10,000 account. Earlier
prices can warm up the holdout indicators, but its returns only begin after the split. Once I inspect
the holdout, repeatedly changing the rule to improve that score would undermine the comparison.

In [ ]:
cut = int(len(bars) * 0.65)
if cut < 40 or len(bars) - cut < 20:
    raise ValueError("Need more history for this development/holdout experiment.")
settings = dict(initial_cash=10000.0, allocation=0.2, fee_bps=1.0, slippage_bps=5.0)
development = backtest(bars.iloc[:cut], candidate, start=30, **settings)
holdout = backtest(bars, candidate, start=cut, **settings)
baseline = backtest(bars, lambda closes: 1, start=cut, **settings)
summary = pd.DataFrame(
    {
        "Development / strategy": development[2],
        "Holdout / strategy": holdout[2],
        "Holdout / buy and hold": baseline[2],
    }
).T
print(provenance, "|", data_label)
display(summary.round(3))
display(holdout[1].head(10))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, layout="constrained")
for name, result in [
    ("Generated strategy" if GENERATE_WITH_LLM else "Authored reference", holdout),
    ("Buy and hold (same entry allocation)", baseline),
]:
    equity = result[0].equity
    axes[0].plot(equity.index, equity, label=name)
    peak = equity.cummax().clip(lower=settings["initial_cash"])
    axes[1].plot(equity.index, 100 * (equity / peak - 1), label=name)
axes[0].set(title=f"Holdout equity — {data_label}", ylabel="Account value ($)")
axes[1].set(ylabel="Drawdown (%)", xlabel="Date")
axes[0].legend()
for ax in axes:
    ax.grid(alpha=0.25)
plt.show()

## 7. Accounting checks and cost sensitivity

I use a small example that I can calculate by hand: a signal from a $10 close buys at the next open
of $20, then exits at $30. With $100 and no costs, the account buys five shares and finishes with $150.
I record the history passed to the strategy to check that it cannot see the next day's close.

I then increase slippage without changing the rule. The Sharpe calculation assumes 252 daily
observations per year and a zero risk-free rate. Dividends, interest, taxes, liquidity limits, partial
fills, and trading halts are outside this backtest's scope.

In [ ]:
tiny = pd.DataFrame(
    {"open": [10.0, 20.0, 30.0], "close": [10.0, 25.0, 30.0]},
    index=pd.date_range("2024-01-02", periods=3),
)
seen = []


def known_signal(closes):
    seen.append(len(closes))
    return 1 if len(closes) == 1 else 0


curve, trades, metrics = backtest(
    tiny,
    known_signal,
    initial_cash=100,
    allocation=1,
    fee_bps=0,
    slippage_bps=0,
    start=1,
)
assert seen == [1, 2]
assert trades.qty.tolist() == [5, 5]
assert trades.fill.tolist() == [20.0, 30.0]
assert metrics["final_equity"] == 150.0
assert (curve.cash >= -1e-8).all()
# Costs paid on entry and exit should lower the result of this known round trip.
with_costs = backtest(
    tiny,
    lambda c: int(len(c) == 1),
    initial_cash=100,
    allocation=1,
    fee_bps=10,
    slippage_bps=10,
    start=1,
)
assert with_costs[2]["final_equity"] < 150
print("Next-open timing, whole-share accounting, and cost checks passed.")

sensitivity = []
for bps in [0, 5, 20, 50]:
    result = backtest(bars, candidate, start=cut, **{**settings, "slippage_bps": bps})
    sensitivity.append({"slippage_bps": bps, **result[2]})
display(pd.DataFrame(sensitivity).round(3))

## 8. Comparing models

For code generation, I care about implementing the requested rule, then latency and cost. A profitable
backtest would not tell me whether the model followed the specification.

`RUN_MODEL_COMPARISON` makes additional paid requests with the same prompt and examples. The listed
models use low reasoning effort and a 3,000-output-token limit. One trial per model is a small check,
not a reliability study. Failed attempts remain in the table, so first-attempt success and success
after repair stay distinguishable.

`output_tokens` includes reasoning tokens; I do not add `reasoning_tokens` again when estimating cost.

In [ ]:
RUN_MODEL_COMPARISON = False
MODELS_TO_COMPARE = ["gpt-6-luna", "gpt-6.2-sol", "gpt-6-astra"]
comparison = pd.DataFrame()
if RUN_MODEL_COMPARISON:
    from openai import OpenAI

    comparison_client = OpenAI(timeout=60.0, max_retries=0)
    logs = []
    for model_name in MODELS_TO_COMPARE:
        _, log = generate_strategy(comparison_client, model_name, SPECIFICATION, CASES)
        logs.append(log)
    comparison = pd.concat(logs, ignore_index=True)
    display(comparison.drop(columns="source"))
else:
    print("Model comparison disabled; no additional API calls.")

### Observed run: November 21, 2026

One live API generation per model used the same specification, API contract, and low reasoning effort.
GPT-6 Astra also completed all notebook cells using synthetic data. No repair was needed.

| Model | Attempts | Seconds | Input / output tokens | Independent checks | Estimated request cost |
|---|---:|---:|---:|---:|---:|
| gpt-6-astra | 1 | 3.23 | 204 / 60 | 150/150 | $0.0050400 |
| gpt-6-luna | 1 | 2.07 | 204 / 101 | 150/150 | $0.0000709 |
| gpt-6.2-sol | 1 | 1.51 | 204 / 57 | 150/150 | $0.0009780 |

All three generated the same decisions and holdout result: **-0.778%** versus **+2.190%** for the
allocation-matched buy-and-hold baseline on this synthetic fixture. Luna returned the same source
as Astra; Sol's source differed but produced the same tested behavior. These are software validation
results, not evidence of a profitable strategy or a reliable speed ranking from one trial.

Costs use reported token counts and published Standard rates, including reasoning output tokens;
they are estimates rather than billing records. Pricing sources:
[Astra](https://developers.openai.com/api/docs/models/gpt-6-astra),
[Luna](https://developers.openai.com/api/docs/models/gpt-6-luna), and
[Sol](https://developers.openai.com/api/docs/models/gpt-6.2-sol).
This result is why I chose Luna as my default for this small function. I keep the same tests regardless of the model.


## 9. Checking the paper connection and previewing an order

Once my paper keys are configured, I set `CONNECT_PAPER = True`. The first cell checks the account
and market clock without sending an order. That connection check also works while the market is closed.
The constructor always uses `paper=True`.

I use a dedicated paper account because the adapter treats **all shares of the selected symbol** as
this strategy's position, including any shares bought manually. For an entry, I set `PAPER_BUDGET`
below. The actual entry budget is capped at the smallest of that amount, 10% of equity, and 98% of cash.
An exit sells the entire whole-share position.

The preview checks the regular session, account permissions, tradability, open orders, the latest
completed trading day, and a recent IEX quote. It returns HOLD when no order is needed. An entry also
requires enough budget for at least one whole share. A cash signal with no position correctly does nothing;
I do not override it just to make a trade appear.

The backtest assumes next-open execution. This paper demo runs when I execute the cell and uses DAY
limit orders near the current quote, so its fills can differ or may not occur at all.

In [ ]:
if CONNECT_PAPER:
    from paper_broker import PaperBroker

    broker = PaperBroker()
    display(broker.connection_status())
else:
    print(
        "Set CONNECT_PAPER = True after saving the Alpaca paper keys and restarting the kernel."
    )

In [ ]:
PAPER_BUDGET = 1000.0  # Maximum entry amount in simulated dollars.
plan = None
if CONNECT_PAPER:
    from paper_broker import PaperBroker

    broker = PaperBroker()
    plan = broker.preview(candidate, symbol=SYMBOL, max_notional=PAPER_BUDGET)
    display(plan)
else:
    print("Paper account connection disabled. Offline tests are complete.")

## 10. Submitting and checking a paper order

After reviewing a BUY or SELL preview, I set `SUBMIT_PAPER_ORDER = True` and run the submission cell.
The adapter checks the plan again immediately. If the quote or account state has changed, I refresh
the preview and review the new order before submitting.

Each order has an ID derived from the strategy source, symbol, completed session, and side. This
helps cell reruns find the existing order. After an ambiguous timeout, the adapter checks that ID
before doing anything else. A canceled or rejected order with the same ID also stops an automatic retry.

An accepted order is not necessarily filled. I check its status below and in Alpaca's Orders dashboard.
The dashboard can also cancel a pending order. This notebook does not run a background trading loop.

In [ ]:
SUBMIT_PAPER_ORDER = False
submission = None
if SUBMIT_PAPER_ORDER:
    if not CONNECT_PAPER or plan is None or plan.get("action") not in ("buy", "sell"):
        raise RuntimeError("Connect and obtain a BUY or SELL preview first.")
    submission = broker.submit(
        candidate, plan, symbol=SYMBOL, max_notional=PAPER_BUDGET
    )
    display(submission)
else:
    print("Submission disabled. No paper order sent.")

In [ ]:
if CONNECT_PAPER and submission:
    display(broker.order_status(submission["client_order_id"]))
# Rerunning this cell refreshes the order status without submitting another order.

## 11. An optional Gradio interface

I added Gradio so I can demonstrate the same workflow without stepping through every notebook cell.
It calls the existing strategy engine and broker adapter rather than maintaining a second implementation.

After running the setup cell, I set `LAUNCH_GRADIO = True` below. The app starts locally at the printed
address, usually `http://127.0.0.1:7860`. It does not create a public share link. Launching it does not
call a model or place an order. If I already generated a strategy in this notebook, the editor starts
with that source; otherwise it starts with the offline reference.

My workflow in the app is:
1. Edit the English rule and its expected examples, then click **Generate strategy (paid API)**.
   **Load offline reference** gives me a free starting point instead.
2. Click **Validate and backtest** to see the holdout metrics, chart, and simulated orders.
3. After saving paper keys and restarting the kernel, enable **Alpaca paper-account access** and click
   **Check paper connection** in the paper-trading tab.
4. Click **Preview paper order**. If it contains a BUY or SELL I want to test, check **I reviewed this
   paper order**, then click **Submit reviewed paper order**. **Refresh order status** checks the result.

The app has its own paper-access checkbox; notebook flags do not silently enable that access.
Changing the source, examples, symbol, or budget invalidates the old preview. The preview is stored per
browser session and is checked again on submission. A closed market or HOLD signal still means no trade.
The keys are read on the server from `.env`; there are no credential fields in the interface.


In [ ]:
LAUNCH_GRADIO = False
if LAUNCH_GRADIO:
    from gradio_app import build_ui

    if "ui" in globals():
        globals()["ui"].close()
    initial_source = candidate.source if "candidate" in globals() else REFERENCE_SOURCE
    ui = build_ui(initial_source=initial_source, model=MODEL)
    ui.queue().launch(server_name="127.0.0.1", share=False, show_error=False)
else:
    print("Gradio is optional. Set LAUNCH_GRADIO = True to open the local app.")

## 12. Saving an experiment

`EXPORT_RESULTS` saves the exact source, prompt, model attempts, settings, input bars, and holdout
results to the Git-ignored `local/` directory. I keep the data because model output and provider history
can change between runs. The export excludes credentials and paper-account responses.

In [ ]:
EXPORT_RESULTS = False
if EXPORT_RESULTS:
    artifact_dir = HERE / "local" / candidate.digest
    artifact_dir.mkdir(parents=True, exist_ok=True)
    (artifact_dir / "strategy.py").write_text(candidate.source, encoding="utf-8")
    manifest = {
        "provenance": provenance,
        "specification": SPECIFICATION,
        "api_contract": API_CONTRACT,
        "source_sha256_prefix": candidate.digest,
        "data": data_label,
        "holdout_start": str(bars.index[cut]),
        "settings": settings,
    }
    (artifact_dir / "experiment.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )
    bars.to_csv(artifact_dir / "bars.csv")
    summary.to_csv(artifact_dir / "metrics.csv")
    holdout[1].to_csv(artifact_dir / "holdout_orders.csv", index=False)
    experiment.to_json(artifact_dir / "model_attempts.json", orient="records", indent=2)
    print("Saved:", artifact_dir)

## 13. Findings and next experiments

My first comparison was useful because the smallest model passed the same checks as the larger ones.
The strategy still lost money on the synthetic holdout. Correct code and a good trading rule are
separate questions, and this project currently establishes the former only for the examples tested.

The main choices I would explain in a walkthrough are the narrow generated-code interface, independent
behavior checks, next-open execution, and the distinction between an accepted order and a filled one.
My next experiments would be a second rule with its own examples and repeated model trials. On November 21, 2026, I verified paper-account authentication, retrieved 501 daily AAPL bars
from IEX, and completed a historical backtest with the generated Luna strategy. The paper preview
returned HOLD because the regular session was closed. An actual paper-order fill is still untested.

For the contribution, I keep just the notebook and its three helper modules. Before opening a PR,
I reset the opt-in switches (including `LAUNCH_GRADIO`) to `False`, run the offline path, and clear all
notebook outputs and execution counts. I stage this folder explicitly to avoid including my lesson edits.
[Ed's contribution guide](https://edwarddonner.com/pr/)

```bash
git switch -c codex/week4-alpaca-codegen
git add week4/community-contributions/SeeBurtOh/alpaca_codegen
git diff --cached --stat
git commit -m "Add Week 4 Alpaca strategy code generator"
git push -u origin codex/week4-alpaca-codegen
```

The PR targets `ed-donner/llm_engineering`'s `main` branch from my fork. I check that the staged diff
and GitHub's **Files changed** contain only these four contribution files before submitting.

### References
- [Alpaca paper-trading environment](https://docs.alpaca.markets/us/docs/paper-trading)
- [Alpaca Python SDK](https://alpaca.markets/sdks/python/trading.html)
- [Order IDs and tracking](https://docs.alpaca.markets/us/docs/working-with-orders)
- [Gradio session state](https://www.gradio.app/guides/state-in-blocks)